# ForgeAI Colab T4 Ollama + ngrok Server

This notebook runs an Ollama-compatible local LLM on a Colab **T4 GPU** and exposes it through an ngrok HTTPS tunnel. ForgeAI can then use the tunnel from your laptop with `--provider ollama --ollama-url <ngrok-url>`.

## Security note

Ollama has no built-in auth. An ngrok URL is a public URL. Keep it private, stop the tunnel when done, and do not use it for sensitive prompts.

## Stay-alive note

Colab disconnects idle runtimes. Keep the browser tab open and check **Runtime → Manage sessions** while ForgeAI is generating; if the runtime dies the tunnel dies with it.


## 1. Confirm the GPU runtime

In Colab, choose **Runtime → Change runtime type → T4 GPU** before running this cell.


In [ ]:
!nvidia-smi


## 2. Install Ollama and ngrok helpers

This installs the Ollama server and `pyngrok`. You need a free ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken. Store it in Colab Secrets as `NGROK_AUTHTOKEN`, or paste it when prompted.


In [ ]:
!apt-get update -qq
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install pyngrok requests


## 3. Start Ollama on all interfaces

Colab needs Ollama listening on `0.0.0.0:11434` so ngrok can forward to it.

**Important env knobs for ForgeAI:**
- `OLLAMA_CONTEXT_LENGTH=8192` — ForgeAI's Verse stage sends long system + knowledge prompts and asks for up to 4096 output tokens. Ollama defaults to `num_ctx=2048`, which silently truncates the JSON response and produces invalid declarations.
- `OLLAMA_KEEP_ALIVE=30m` — keeps the model loaded between ForgeAI's per-stage calls so you don't pay first-token latency every time.
- `OLLAMA_FLASH_ATTENTION=1` — meaningful speedup on T4 for 7B models.


In [ ]:
import os
import subprocess
import time

ollama_env = os.environ.copy()
ollama_env["OLLAMA_HOST"] = "0.0.0.0:11434"
ollama_env["OLLAMA_ORIGINS"] = "*"
ollama_env["OLLAMA_CONTEXT_LENGTH"] = "8192"
ollama_env["OLLAMA_KEEP_ALIVE"] = "30m"
ollama_env["OLLAMA_FLASH_ATTENTION"] = "1"

server = subprocess.Popen(["ollama", "serve"], env=ollama_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(5)
print("Ollama server started with pid", server.pid)
!curl -s http://127.0.0.1:11434/api/tags


## 4. Pull a T4-friendly model

**Default: `qwen2.5-coder:7b-instruct`.** This is the coder-tuned variant of Qwen2.5 7B. For ForgeAI's Verse AST JSON output it is noticeably more reliable than the plain `qwen2.5:7b-instruct` chat model, because the coder model is trained specifically on structured code/JSON.

If you want the chat model instead, set `MODEL = "qwen2.5:7b-instruct"`. Larger models (14B+) will likely OOM on T4's 16 GB.


In [ ]:
MODEL = "qwen2.5-coder:7b-instruct"
!ollama pull {MODEL}
!ollama list


## 5. Smoke test the model locally inside Colab

This test mimics ForgeAI's actual call shape: `format: "json"`, low temperature, long system prompt, and a request to produce a Verse-module-shaped JSON object with a discriminated-union declaration. If this returns malformed JSON or a payload missing `declarations[0].kind`, the full ForgeAI run will fail the same way.


In [ ]:
import json
import requests

system_prompt = (
    "Return ONLY valid JSON matching this schema (no prose, no markdown):\n"
    "{\n"
    "  \"kind\": \"module\",\n"
    "  \"name\": string,\n"
    "  \"imports\": [{\"kind\":\"import\",\"path\":string}],\n"
    "  \"declarations\": [\n"
    "    {\"kind\":\"class\",\"name\":string,\"extends\":string,\"fields\":[],\"methods\":[\n"
    "      {\"kind\":\"function\",\"name\":string,\"params\":[],\"returnType\":string,\"attributes\":[string],\"body\":[{\"kind\":\"statement\",\"code\":string}]}\n"
    "    ]}\n"
    "  ]\n"
    "}\n"
    "Every object MUST include its 'kind' discriminator."
)
user_msg = "Generate a Verse module named tycoon_economy_manager with one class tycoon_economy_manager extending creative_device that has a single OnBegin method."

payload = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_msg},
    ],
    "stream": False,
    "format": "json",
    "options": {"num_ctx": 8192, "temperature": 0.2, "num_predict": 1024},
}
response = requests.post("http://127.0.0.1:11434/api/chat", json=payload, timeout=300)
print("HTTP", response.status_code)
data = response.json()
content = data.get("message", {}).get("content", "")
print("--- raw content ---")
print(content[:2000])
print("--- parsed sanity check ---")
parsed = json.loads(content)
assert parsed.get("kind") == "module", "missing top-level kind=module"
assert isinstance(parsed.get("declarations"), list) and parsed["declarations"], "empty declarations"
decl0 = parsed["declarations"][0]
assert decl0.get("kind") in ("class", "function"), f"bad declarations[0].kind: {decl0.get('kind')}"
print("OK: declarations[0].kind =", decl0["kind"])


## 6. Open an ngrok tunnel

Copy the printed public URL. Use it as ForgeAI's `--ollama-url`. Keep this Colab runtime alive while generating.

**ngrok-free interstitial:** the first request from a new IP to a free `*.ngrok-free.app` URL gets an HTML browser-warning page instead of the JSON you expect. ForgeAI's HTTP client will fail to parse that as JSON. We add an `ngrok-skip-browser-warning` request header rule below; you should also send the same header from the client if you ever switch HTTP libraries. (Reserved domains on a paid plan avoid this entirely.)


In [ ]:
import getpass
from pyngrok import ngrok, conf

try:
    from google.colab import userdata
    ngrok_token = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    ngrok_token = None

if not ngrok_token:
    ngrok_token = getpass.getpass("Paste NGROK_AUTHTOKEN: ")

conf.get_default().auth_token = ngrok_token
try:
    ngrok.kill()
except Exception:
    pass

tunnel = ngrok.connect(
    11434,
    "http",
    request_header_add=["ngrok-skip-browser-warning:true"],
)
OLLAMA_PUBLIC_URL = tunnel.public_url
print("OLLAMA_PUBLIC_URL=" + OLLAMA_PUBLIC_URL)
print("Model=" + MODEL)


## 7. Test the public tunnel from Colab


In [ ]:
headers = {"ngrok-skip-browser-warning": "true"}
public_tags = requests.get(OLLAMA_PUBLIC_URL + "/api/tags", headers=headers, timeout=30)
print(public_tags.status_code)
print(public_tags.text[:1000])


## 8. Use this server from ForgeAI

On your local machine, run:

```bash
export FORGEAI_OLLAMA_BASE_URL="https://YOUR-NGROK-URL.ngrok-free.app"
uefn-ai doctor --provider ollama --model qwen2.5-coder:7b-instruct --ollama-url "$FORGEAI_OLLAMA_BASE_URL"

uefn-ai create "A compact lumber tycoon for 4 players with one upgrade lane and worker automation." \
  --provider ollama \
  --model qwen2.5-coder:7b-instruct \
  --ollama-url "$FORGEAI_OLLAMA_BASE_URL" \
  --genre tycoon \
  --template tycoon/lumber-mill \
  --seed 101 \
  --out ./output/colab-t4-lumber \
  --budget 0.01 \
  --zip
```

The budget can be very low because local/Ollama calls are priced at `$0` in ForgeAI's ledger. It is still useful as a guardrail if fallback providers are enabled.

If you previously hit `$.declarations.0: Invalid input` on the Verse stage with `qwen2.5:7b-instruct`, the combination of (a) `qwen2.5-coder:7b-instruct`, (b) `OLLAMA_CONTEXT_LENGTH=8192`, and (c) the discriminated-union schema/error improvements in ForgeAI should resolve it. If it still fails, the new error message will list the actual missing fields per branch — share that and we can iterate.


## 9. Stop the tunnel when done


In [ ]:
# Run this when finished.
# ngrok.kill()
# server.terminate()
